In [5]:
from google.colab import files
uploaded = files.upload()

Saving car_prices_final_preprocessed (3).csv to car_prices_final_preprocessed (3).csv


In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.6 MB/s eta 0:00:00


In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("ADVANCED CAR PRICE PREDICTION - NO DATA LEAKAGE")
print("="*80)

# ============================================
# STEP 1: Load and Initial Cleaning
# ============================================
df = pd.read_csv('car_prices_final_preprocessed.csv')
print(f"\nInitial dataset shape: {df.shape}")

# Remove dummy prices
dummy_prices = [123456, 111111, 99999, 123450]
df = df[~df['price'].isin(dummy_prices)]
df = df[~((df['price'] % 10000 == 0) & (df['price'] > 200000))]

# ============================================
# STEP 2: CRITICAL - Feature-Based Segmentation (NO DATA LEAKAGE)
# ============================================
print("\n" + "="*80)
print("FEATURE-BASED MARKET SEGMENTATION (No Price Used!)")
print("="*80)

def assign_segment_from_features(row):
    """
    Assign segment based ONLY on features we know at prediction time.
    NO price information used!
    """
    score = 0

    # Brand positioning (most important)
    brand_percentile = (row['brand_encoded'] - df['brand_encoded'].min()) / (df['brand_encoded'].max() - df['brand_encoded'].min())
    if brand_percentile > 0.7:
        score += 40  # Luxury brand
    elif brand_percentile > 0.4:
        score += 20  # Mid-tier brand
    else:
        score += 0   # Budget brand

    # Power/Engine (puissance fiscale)
    if row['puissance_fiscale'] > 10:
        score += 25
    elif row['puissance_fiscale'] > 6:
        score += 10

    # Age of car
    if row['car_age'] <= 3:
        score += 20
    elif row['car_age'] <= 7:
        score += 10
    elif row['car_age'] > 12:
        score -= 15

    # New car indicator
    if row['is_new'] == 1:
        score += 15

    # Fuel type (electric/hybrid typically higher segment)
    if row.get('fuel_electrique', 0) == 1:
        score += 10
    elif row.get('fuel_hybride', 0) == 1:
        score += 5

    # Transmission
    if row['transmission'] == 1:  # Automatic
        score += 5

    # Body type encoding (proxy for segment)
    body_percentile = (row['body_type_encoded'] - df['body_type_encoded'].min()) / (df['body_type_encoded'].max() - df['body_type_encoded'].min())
    if body_percentile > 0.6:
        score += 10

    # Assign segment based on score
    if score >= 60:
        return 'luxury'
    elif score >= 30:
        return 'mid'
    else:
        return 'budget'

df['market_segment'] = df.apply(assign_segment_from_features, axis=1)

print("\nSegment distribution (feature-based):")
print(df['market_segment'].value_counts())
print("\nSegment vs actual price (validation):")
print(df.groupby('market_segment')['price'].describe()[['mean', '25%', '75%']])

# Verify no price leakage
print("\n✓ Segmentation uses ONLY features available at prediction time")
print("✓ No price information used in segment assignment")

# ============================================
# STEP 3: Data Quality Flags (NO PRICE)
# ============================================
print("\n" + "="*80)
print("DATA QUALITY ANALYSIS (Feature-Based)")
print("="*80)

def calculate_quality_score_no_price(row):
    """Quality score without using price"""
    score = 100

    # Suspicious age + mileage combinations
    if row['car_age'] < 3 and row['kilometrage'] > 100000:
        score -= 30  # Too much mileage for new car

    if row['car_age'] > 15 and row['kilometrage'] < 50000:
        score -= 20  # Suspiciously low mileage for old car

    # Extreme mileage
    if row['kilometrage'] > 350000:
        score -= 25

    # Very old car
    if row['car_age'] > 20:
        score -= 15

    # Inconsistent new flag
    if row['is_new'] == 1 and row['car_age'] > 1:
        score -= 40

    if row['is_new'] == 1 and row['kilometrage'] > 5000:
        score -= 30

    return max(score, 0)

df['quality_score'] = df.apply(calculate_quality_score_no_price, axis=1)

print(f"Quality score distribution:")
print(df['quality_score'].describe())

# Remove very suspicious listings
very_suspicious = df['quality_score'] < 40
print(f"Removing {very_suspicious.sum()} very low quality listings")
df_clean = df[~very_suspicious].copy()

print(f"Dataset after quality filtering: {df_clean.shape}")

# ============================================
# STEP 4: Advanced Feature Engineering
# ============================================
print("\n" + "="*80)
print("ADVANCED FEATURE ENGINEERING")
print("="*80)

# Basic interactions
df_clean['power_age_interaction'] = df_clean['puissance_fiscale'] * df_clean['car_age']
df_clean['km_per_year'] = df_clean['kilometrage'] / (df_clean['car_age'] + 1)
df_clean['puissance_fiscale_sq'] = df_clean['puissance_fiscale'] ** 2
df_clean['car_age_sq'] = df_clean['car_age'] ** 2
df_clean['car_age_cube'] = df_clean['car_age'] ** 3
df_clean['km_log'] = np.log1p(df_clean['kilometrage'])
df_clean['km_sqrt'] = np.sqrt(df_clean['kilometrage'])

# NEW: Depreciation curve features
df_clean['exp_depreciation'] = np.exp(-0.12 * df_clean['car_age'])
df_clean['sqrt_depreciation'] = np.sqrt(df_clean['car_age'] + 1)

# Usage patterns
df_clean['annual_km'] = df_clean['kilometrage'] / (df_clean['car_age'] + 0.5)
df_clean['low_usage'] = (df_clean['annual_km'] < 10000).astype(int)
df_clean['normal_usage'] = ((df_clean['annual_km'] >= 10000) & (df_clean['annual_km'] <= 20000)).astype(int)
df_clean['high_usage'] = (df_clean['annual_km'] > 25000).astype(int)

# Age categories
df_clean['age_bracket'] = pd.cut(df_clean['car_age'], bins=[0, 3, 7, 12, 25], labels=['new', 'recent', 'mature', 'old'])
df_clean['age_bracket_encoded'] = df_clean['age_bracket'].cat.codes

# Brand-feature interactions
df_clean['brand_power_interaction'] = df_clean['brand_encoded'] * df_clean['puissance_fiscale']
df_clean['brand_age_interaction'] = df_clean['brand_encoded'] * df_clean['car_age']
df_clean['brand_transmission'] = df_clean['brand_encoded'] * df_clean['transmission']

# Body-feature interactions
df_clean['body_power_interaction'] = df_clean['body_type_encoded'] * df_clean['puissance_fiscale']
df_clean['body_age_interaction'] = df_clean['body_type_encoded'] * df_clean['car_age']

# Segment indicators (as features, not for splitting)
df_clean['luxury_indicator'] = (df_clean['market_segment'] == 'luxury').astype(int)
df_clean['budget_indicator'] = (df_clean['market_segment'] == 'budget').astype(int)

# Condition proxies
df_clean['relative_mileage'] = df_clean['kilometrage'] / (df_clean['car_age'] + 1)
df_clean['low_mileage_indicator'] = (df_clean['kilometrage'] < 80000).astype(int)
df_clean['high_mileage_indicator'] = (df_clean['kilometrage'] > 180000).astype(int)

# Power-to-age ratio
df_clean['power_to_age'] = df_clean['puissance_fiscale'] / (df_clean['car_age'] + 1)

# Model rarity (how common is this model?)
model_counts = df_clean.groupby('model_encoded').size()
df_clean['model_frequency'] = df_clean['model_encoded'].map(model_counts)
df_clean['rare_model'] = (df_clean['model_frequency'] < 10).astype(int)

# Brand rarity
brand_counts = df_clean.groupby('brand_encoded').size()
df_clean['brand_frequency'] = df_clean['brand_encoded'].map(brand_counts)

# Complex interaction: depreciation adjusted for brand quality
df_clean['brand_adjusted_depreciation'] = df_clean['brand_encoded'] * df_clean['exp_depreciation']

# Mileage severity by age
df_clean['mileage_severity'] = df_clean['kilometrage'] / ((df_clean['car_age'] + 1) * 15000)  # 15k km/year baseline

num_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[num_cols] = df_clean[num_cols].fillna(0)

# Fill categorical NaN with the mode (most common)
cat_cols = df_clean.select_dtypes(include=['category', 'object']).columns
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
print(f"Total features: {df_clean.shape[1] - 1}")
print(f"Added 30+ engineered features")

# ============================================
# STEP 5: Segment-Specific Outlier Removal
# ============================================
print("\n" + "="*80)
print("SEGMENT-SPECIFIC OUTLIER REMOVAL")
print("="*80)

def remove_segment_outliers(data):
    result = pd.DataFrame()

    for segment in ['budget', 'mid', 'luxury']:
        seg_data = data[data['market_segment'] == segment].copy()
        if len(seg_data) == 0:
            continue

        initial_count = len(seg_data)

        # Remove price outliers per segment
        if segment == 'budget':
            q_low, q_high = seg_data['price'].quantile([0.005, 0.995])
        elif segment == 'mid':
            q_low, q_high = seg_data['price'].quantile([0.01, 0.99])
        else:
            q_low, q_high = seg_data['price'].quantile([0.02, 0.98])

        seg_data = seg_data[(seg_data['price'] >= q_low) & (seg_data['price'] <= q_high)]

        removed = initial_count - len(seg_data)
        print(f"{segment.capitalize()}: kept {len(seg_data)}/{initial_count} ({removed} removed)")

        result = pd.concat([result, seg_data])

    return result

df_final = remove_segment_outliers(df_clean)
print(f"\nFinal dataset: {df_final.shape}")

# ============================================
# STEP 6: Prepare Training Data
# ============================================
print("\n" + "="*80)
print("PREPARING TRAINING DATA")
print("="*80)

# Drop non-feature columns
cols_to_drop = ['price', 'market_segment', 'age_bracket', 'quality_score','price_per_fiscal']
cols_to_drop = [col for col in cols_to_drop if col in df_final.columns]

X = df_final.drop(cols_to_drop, axis=1)
y = df_final['price']
segments = df_final['market_segment']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Stratified split
strata = df_final['market_segment'].astype(str) + '_' + pd.qcut(y, q=3, labels=['low', 'mid', 'high'], duplicates='drop').astype(str)

X_train, X_test, y_train, y_test, seg_train, seg_test = train_test_split(
    X, y, segments, test_size=0.2, random_state=42, stratify=strata
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# ============================================
# STEP 7: Train Segment-Specific Models
# ============================================
print("\n" + "="*80)
print("TRAINING SEGMENT-SPECIFIC MODELS")
print("="*80)

segment_models = {}
segment_results = {}

for segment in ['budget', 'mid', 'luxury']:
    print(f"\n{'='*80}")
    print(f"Training {segment.upper()} segment model")
    print(f"{'='*80}")

    train_mask = seg_train == segment
    test_mask = seg_test == segment

    X_train_seg = X_train[train_mask]
    y_train_seg = y_train[train_mask]
    X_test_seg = X_test[test_mask]
    y_test_seg = y_test[test_mask]

    print(f"Train samples: {len(X_train_seg)}")
    print(f"Test samples: {len(X_test_seg)}")

    if len(X_train_seg) < 50:
        print(f"⚠ Too few samples, skipping")
        continue

    # Segment-specific models
    if segment == 'budget':
        model = lgb.LGBMRegressor(
            n_estimators=600, learning_rate=0.015, max_depth=6,
            num_leaves=30, min_child_samples=15, subsample=0.8,
            colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=2.0,
            random_state=42, verbose=-1, n_jobs=-1
        )
    elif segment == 'mid':
        model = xgb.XGBRegressor(
            n_estimators=700, learning_rate=0.012, max_depth=5,
            min_child_weight=3, subsample=0.75, colsample_bytree=0.75,
            reg_alpha=1.2, reg_lambda=2.2, gamma=0.3,
            random_state=42, n_jobs=-1
        )
    else:  # luxury
        model = CatBoostRegressor(
            iterations=1000, learning_rate=0.008, depth=6,
            l2_leaf_reg=6.0, random_state=42, verbose=0
        )

    model.fit(X_train_seg, y_train_seg)

    y_train_pred_seg = model.predict(X_train_seg)
    y_test_pred_seg = model.predict(X_test_seg)

    train_mae = mean_absolute_error(y_train_seg, y_train_pred_seg)
    test_mae = mean_absolute_error(y_test_seg, y_test_pred_seg)
    train_r2 = r2_score(y_train_seg, y_train_pred_seg)
    test_r2 = r2_score(y_test_seg, y_test_pred_seg)

    pct_errors = np.abs((y_test_pred_seg - y_test_seg) / y_test_seg * 100)

    segment_models[segment] = model
    segment_results[segment] = {
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'test_pred': y_test_pred_seg,
        'test_actual': y_test_seg,
        'pct_errors': pct_errors
    }

    print(f"\nResults:")
    print(f"  Train MAE: {train_mae:,.0f} TND")
    print(f"  Test MAE:  {test_mae:,.0f} TND")
    print(f"  Train R²: {train_r2:.4f}")
    print(f"  Test R²:  {test_r2:.4f}")
    print(f"  Within 10%: {(pct_errors <= 10).mean()*100:.1f}%")
    print(f"  Within 20%: {(pct_errors <= 20).mean()*100:.1f}%")

# ============================================
# STEP 8: Prediction Function (How to Use in Production)
# ============================================
print(f"\n{'='*80}")
print("CREATING PRODUCTION PREDICTION FUNCTION")
print(f"{'='*80}")

def predict_car_price(car_features_dict):
    """
    Predict car price for a new car.

    Parameters:
    -----------
    car_features_dict : dict
        Dictionary with car features (brand_encoded, car_age, etc.)

    Returns:
    --------
    float : predicted price
    str : segment used for prediction
    """
    # Convert to DataFrame
    car_df = pd.DataFrame([car_features_dict])

    # Determine segment from features (NO PRICE NEEDED!)
    segment = assign_segment_from_features(car_df.iloc[0])

    # Use appropriate model
    if segment in segment_models:
        model = segment_models[segment]
        prediction = model.predict(car_df)[0]
    else:
        # Fallback to global model if segment model not available
        prediction = global_model.predict(car_df)[0]

    # Apply business rules
    prediction = apply_business_rules_single(prediction, car_df.iloc[0])

    return prediction, segment

def apply_business_rules_single(prediction, features):
    """Apply business rules to a single prediction"""

    # New cars minimum
    if features.get('is_new', 0) == 1:
        prediction = max(prediction, 25000)

    # Very old cars maximum
    if features['car_age'] > 15:
        if features['brand_encoded'] < df_final['brand_encoded'].quantile(0.7):
            prediction = min(prediction, 60000)

    # High mileage cap
    if features['kilometrage'] > 200000:
        prediction = min(prediction, 50000)

    # Luxury brand floor
    if features['brand_encoded'] > df_final['brand_encoded'].quantile(0.85):
        if features['car_age'] < 20:
            prediction = max(prediction, 35000)

    return prediction

print("✓ Production prediction function created")
print("\nExample usage:")
print("prediction, segment = predict_car_price({")
print("    'brand_encoded': 0.5,")
print("    'car_age': 5,")
print("    'kilometrage': 80000,")
print("    'puissance_fiscale': 7,")
print("    'transmission': 1,")
print("    '...' : '...'")
print("})")

# ============================================
# STEP 9: Global Model + Combined Predictions
# ============================================
print(f"\n{'='*80}")
print("TRAINING GLOBAL MODEL")
print(f"{'='*80}")

global_model = lgb.LGBMRegressor(
    n_estimators=600, learning_rate=0.015, max_depth=6,
    num_leaves=25, min_child_samples=20, subsample=0.75,
    colsample_bytree=0.75, reg_alpha=1.5, reg_lambda=2.5,
    random_state=42, verbose=-1, n_jobs=-1
)

global_model.fit(X_train, y_train)
y_test_pred_global = global_model.predict(X_test)

# Combined segment predictions
y_test_pred_combined = np.zeros(len(y_test))
for segment in ['budget', 'mid', 'luxury']:
    if segment not in segment_models:
        continue
    test_mask = seg_test == segment
    if test_mask.sum() > 0:
        y_test_pred_combined[test_mask] = segment_results[segment]['test_pred']

# Apply business rules
def apply_business_rules_batch(predictions, features, segments):
    adjusted = predictions.copy()
    for i in range(len(predictions)):
        adjusted[i] = apply_business_rules_single(adjusted[i], features.iloc[i])
    return adjusted

y_test_pred_final = apply_business_rules_batch(y_test_pred_combined, X_test.reset_index(drop=True), seg_test)

# ============================================
# STEP 10: Final Results
# ============================================
print(f"\n{'='*80}")
print("FINAL RESULTS COMPARISON")
print(f"{'='*80}")

# Calculate metrics
global_mae = mean_absolute_error(y_test, y_test_pred_global)
global_r2 = r2_score(y_test, y_test_pred_global)
global_pct = np.abs((y_test_pred_global - y_test) / y_test * 100)

combined_mae = mean_absolute_error(y_test, y_test_pred_combined)
combined_r2 = r2_score(y_test, y_test_pred_combined)
combined_pct = np.abs((y_test_pred_combined - y_test) / y_test * 100)

final_mae = mean_absolute_error(y_test, y_test_pred_final)
final_r2 = r2_score(y_test, y_test_pred_final)
final_pct = np.abs((y_test_pred_final - y_test) / y_test * 100)

comparison = pd.DataFrame({
    'Model': ['Original', 'Global Model', 'Segment Models', 'With Business Rules'],
    'MAE (TND)': [8688, global_mae, combined_mae, final_mae],
    'R²': [0.9116, global_r2, combined_r2, final_r2],
    'Within 10%': ['51.2%', f"{(global_pct <= 10).mean()*100:.1f}%",
                   f"{(combined_pct <= 10).mean()*100:.1f}%", f"{(final_pct <= 10).mean()*100:.1f}%"],
    'Max Error': ['276%', f"{global_pct.max():.1f}%",
                  f"{combined_pct.max():.1f}%", f"{final_pct.max():.1f}%"]
})

print("\n" + comparison.to_string(index=False))

print(f"\n🎉 Improvement: {((8688 - final_mae) / 8688 * 100):.1f}% MAE reduction")
print(f"🎉 MAE reduced by: {8688 - final_mae:,.0f} TND")

print(f"\nDetailed Final Metrics:")
print(f"  Mean Absolute Error: {final_mae:,.0f} TND")
print(f"  Median Absolute Error: {np.median(np.abs(y_test_pred_final - y_test)):,.0f} TND")
print(f"  Within 5k TND:  {(np.abs(y_test_pred_final - y_test) <= 5000).mean()*100:.1f}%")
print(f"  Within 10k TND: {(np.abs(y_test_pred_final - y_test) <= 10000).mean()*100:.1f}%")
print(f"  Within 15k TND: {(np.abs(y_test_pred_final - y_test) <= 15000).mean()*100:.1f}%")

# ============================================
# STEP 11: Save Everything
# ============================================
import pickle

print(f"\n{'='*80}")
print("SAVING MODELS AND METADATA")
print(f"{'='*80}")

# Save models
for segment, model in segment_models.items():
    with open(f'model_segment_{segment}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f"✓ Saved model_segment_{segment}.pkl")

with open('model_global_fallback.pkl', 'wb') as f:
    pickle.dump(global_model, f)
print(f"✓ Saved model_global_fallback.pkl")

# Save metadata for production use
metadata = {
    'brand_min': df['brand_encoded'].min(),
    'brand_max': df['brand_encoded'].max(),
    'brand_q70': df['brand_encoded'].quantile(0.7),
    'brand_q85': df['brand_encoded'].quantile(0.85),
    'segment_assignment_function': 'assign_segment_from_features'
}

with open('model_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print(f"✓ Saved model_metadata.pkl")

print(f"\n{'='*80}")
print("✅ COMPLETE - NO DATA LEAKAGE!")
print(f"{'='*80}")
print("\n✓ Segmentation based on features only (brand, age, power, etc.)")
print("✓ Can predict segment for new cars WITHOUT knowing price")
print("✓ Improved precision with segment-specific models")
print("✓ Business rules applied for edge cases")

ADVANCED CAR PRICE PREDICTION - NO DATA LEAKAGE

Initial dataset shape: (5065, 14)

FEATURE-BASED MARKET SEGMENTATION (No Price Used!)

Segment distribution (feature-based):
market_segment
budget    3014
mid       1813
luxury     144
Name: count, dtype: int64

Segment vs actual price (validation):
                         mean       25%       75%
market_segment                                   
budget           49031.263769   32500.0   59500.0
luxury          226385.729167  177625.0  263485.0
mid             102915.532819   66950.0  129900.0

✓ Segmentation uses ONLY features available at prediction time
✓ No price information used in segment assignment

DATA QUALITY ANALYSIS (Feature-Based)
Quality score distribution:
count    4971.0
mean      100.0
std         0.0
min       100.0
25%       100.0
50%       100.0
75%       100.0
max       100.0
Name: quality_score, dtype: float64
Removing 0 very low quality listings
Dataset after quality filtering: (4971, 16)

ADVANCED FEATURE ENGINEE